In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

In [ ]:
net=nn.Sequential(nn.Flatten(),
                  nn.Linear(784,256),
                  nn.ReLU(),
                  nn.Dropout(dropout1),
                  nn.Linear(256,256),
                  nn.ReLU(),
                  nn.Dropout(dropout2),
                  nn.Linear(256,10),
                 )

def init_weights(m):
    if type(m)==nn.Linear:
        nn.init.normal_(m.weight,std=0.01)
net.apply(init_weights);

In [ ]:


def accuracy(y_hat,y):
    if len(y_hat.shape)>1 and y_hat.shape[1]>1:
        y_hat=y_hat.argmax(axis=1)
    cmp=y_hat.type(y.dtype)==y
    return float(cmp.type(y.dtype).sum())


def evaluate_accuracy(net,data_iter):
    if isinstance(net,torch.nn.Module):
        net.eval()     #评估模式，不会有Dropout,准确率稳定
    metric=Accumulator(2)
    with torch.no_grad():
        for X,y in data_iter:
            metric.add(accuracy(net(X),y),y.numel())
    return metric[0]/metric[1]

def train_epoch_ch3(net, train_iter,trainer,loss):
    matric=Accumulator(3) #loss accuracy y.numel
    if isinstance(net,torch.nn.Module):
        net.train()
    for X,y in train_iter:
        y_hat=net(X)
        l=loss(y_hat,y)

        trainer.zero_grad()
        l.mean().backward()
        trainer.step()
        matric.add(float(l.sum()),accuracy(y_hat,y),y.numel())
    return matric[0]/matric[2],matric[1]/matric[2]
def train_ch3(net,train_iter,test_iter,loss,num_epochs,trainer):
    animator = d2l.Animator(xlabel='epoch', xlim=[1, num_epochs], ylim=[0.3, 0.9],
                        legend=['train loss', 'train acc', 'test acc'])
    for epoch in range (num_epochs):
        train_metrics=train_epoch_ch3(net, train_iter, trainer,loss)
        test_acc=evaluate_accuracy(net, test_iter)
        animator.add(epoch+1,train_metrics+(test_acc,))
    train_loss,train_acc=train_metrics
    assert train_loss<0.5,train_loss
    assert train_acc<=1 and train_acc >0.7,train_acc
    assert test_acc<=1 and test_acc >0.7 ,test_acc
num_epochs=10
train_ch3(net ,train_iter, test_iter,loss,num_epochs,trainer)